In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

causas = [
    'Desperfectos mecánicos',
    'Exceso de velocidad',
    'Estado de ebriedad',
    'Invasión de carril',
    'No guardar distancia'
]
vehiculos = ['Motocicleta', 'Automóvil', 'Camión', 'Bus']
departamentos = ['Managua', 'Chinandega', 'León', 'Masaya', 'Estelí', 'Matagalpa']
sexos = ['Masculino', 'Femenino']

n_registros = 600

df_base = pd.DataFrame({
    'id_accidente': range(1, n_registros + 1),
    'fecha': pd.date_range(start='2025-01-01', periods=n_registros, freq='12H').strftime('%Y-%m-%d'),
    'hora': [f"{h:02d}:00" for h in np.random.randint(0, 24, n_registros)],
    'departamento': np.random.choice(departamentos, n_registros, p=[0.35, 0.15, 0.15, 0.15, 0.10, 0.10]),
    'causa_principal': np.random.choice(causas, n_registros),
    'tipo_vehiculo': np.random.choice(vehiculos, n_registros, p=[0.45, 0.30, 0.15, 0.10]),
    'sexo_conductor': np.random.choice(sexos, n_registros, p=[0.75, 0.25]),
    'lesionados': np.random.poisson(lam=1.0, size=n_registros),
    'fallecidos': np.random.choice([0, 1, 2], n_registros, p=[0.85, 0.12, 0.03])
})

df_base.loc[10:14, 'causa_principal'] = np.nan
df_base = pd.concat([df_base, df_base.iloc[[0, 1]]], ignore_index=True)

df_base.to_csv('accidentes_transito.csv', index=False)

df_limpio = df_base.drop_duplicates().copy()
df_limpio['causa_principal'] = df_limpio['causa_principal'].fillna('No especificada')
df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'])
df_limpio['total_victimas'] = df_limpio['lesionados'] + df_limpio['fallecidos']

def clasificar_horario(hora_str):
    h = int(str(hora_str).split(':')[0])
    if 0 <= h < 6:
        return 'Madrugada'
    elif 6 <= h < 12:
        return 'Mañana'
    elif 12 <= h < 18:
        return 'Tarde'
    else:
        return 'Noche'

df_limpio['rango_horario'] = df_limpio['hora'].apply(clasificar_horario)

df_limpio.to_csv('accidentes_transito_limpio.csv', index=False)
print("Archivos creados correctamente.")

### Fase 1: Exploración Inicial de los Datos

#### Descripción General del Dataset
El conjunto de datos `accidentes_transito.csv` registra un total de **602 incidentes viales** ocurridos durante el año 2025. Contiene **8 variables** principales que combinan información temporal (`fecha`, `hora`), geográfica (`departamento`), causas del incidente (`causa_principal`), tipos de transporte involucrados (`tipo_vehiculo`) e indicadores numéricos de impacto social (`lesionados`, `fallecidos`).

#### Preguntas de Investigación
1. ¿Cuál es la causa principal en los accidentes de tránsito?
2. ¿Qué tipo de vehículo está involucrado en la mayor cantidad de personas fallecidas?
3. ¿Cual es el departamento mas afectado con victimas mortales?
4. ¿Cual es el horario donde mas se registran los incidentes viales?
5. ¿Cual es la cantidad de accidentes fatales con mujeres como conductoras?

In [ ]:
import pandas as pd

df = pd.read_csv('accidentes_transito.csv')

df = df.drop_duplicates()
df['causa_principal'] = df['causa_principal'].fillna('No especificada')
df['fecha'] = pd.to_datetime(df['fecha'])

df['total_victimas'] = df['lesionados'] + df['fallecidos']

def clasificar_horario(hora_str):
    h = int(str(hora_str).split(':')[0])
    if 0 <= h < 6:
        return 'Madrugada'
    elif 6 <= h < 12:
        return 'Mañana'
    elif 12 <= h < 18:
        return 'Tarde'
    else:
        return 'Noche'

df['rango_horario'] = df['hora'].apply(clasificar_horario)

df.to_csv('accidentes_transito_limpio.csv', index=False)

print("=== VERIFICACIÓN DE LA LIMPIEZA ===")
print("Filas resultantes:", len(df))
print("Duplicados restantes:", df.duplicated().sum())
print("Nulos en causa_principal:", df['causa_principal'].isnull().sum())
print("\nMuestra de datos procesados:")
print(df[['hora', 'rango_horario', 'lesionados', 'fallecidos', 'total_victimas']].head())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('accidentes_transito_limpio.csv')

print("==================================================")
print("     FASE 3: ANÁLISIS EXPLORATORIO DE DATOS (EDA) ")
print("==================================================")

print("\n--- 1. ESTADÍSTICAS GENERALES DE LESIONADOS Y FALLECIDOS ---")
print(f"Lesionados - Promedio: {df['lesionados'].mean():.2f}")
print(f"Lesionados - Mediana: {df['lesionados'].median():.2f}")
print(f"Lesionados - Mínimo: {df['lesionados'].min()} | Máximo: {df['lesionados'].max()}")
print(f"Lesionados - Desviación Estándar: {df['lesionados'].std():.2f}")

print(f"\nFallecidos - Promedio: {df['fallecidos'].mean():.2f}")
print(f"Fallecidos - Mediana: {df['fallecidos'].median():.2f}")
print(f"Fallecidos - Mínimo: {df['fallecidos'].min()} | Máximo: {df['fallecidos'].max()}")
print(f"Fallecidos - Desviación Estándar: {df['fallecidos'].std():.2f}")

print("\n--- 2. CONTEO DE ACCIDENTES POR CAUSA PRINCIPAL ---")
print(df['causa_principal'].value_counts())

print("\n--- 3. TOTAL DE FALLECIDOS POR TIPO DE VEHÍCULO ---")
print(df.groupby('tipo_vehiculo')['fallecidos'].sum().sort_values(ascending=False))

print("\n--- 4. VÍCTIMAS MORTALES POR DEPARTAMENTO ---")
print(df.groupby('departamento')['fallecidos'].sum().sort_values(ascending=False))

print("\n--- 5. INCIDENTES VIALES POR RANGO HORARIO ---")
print(df['rango_horario'].value_counts())

print("\n--- 6. ACCIDENTES FATALES CON CONDUCTORAS MUJERES ---")
mujeres_fatales = df[(df['sexo_conductor'] == 'Femenino') & (df['fallecidos'] > 0)]
print(f"Total de accidentes fatales con conductoras mujeres: {len(mujeres_fatales)}")

### Fase 3: Análisis Exploratorio de Datos (EDA)

#### Resumen Estadístico Descriptivo
- **Lesionados:** Promedio de 1.03 personas por accidente, equivalente a 1 lesionado por accidente y la cantidad máxima registrada es de 4 lesionados en 1 mismo accidente.
- **Fallecidos:** Promedio de 0.18 muertes por suceso, equivalente a 1 fallecido por accidente y la cantidad máxima registrada es de 2 fallecidos en un mismo accidente.

#### Los 5 Hallazgos Clave
1. **Causa Principal:** No guardar la distancia es la principal causa con 124 accidentes. La segunda causa más reportada fue los desperfectos mecanicos (123) y la tercera causa de accidentes fue el conducir a exceso de velocidad (122).
2. **Tipo de Vehículo:** Las motocicletas representan el vehículo con mayor impacto en mortalidad, sumando 55 fallecidos del total registrado.
3. **Departamento:** El departamento de Managua es el que tiene la mayor cantidad de accidentes con víctimas fatales, alcanzando 43 víctimas fallecidas.
4. **Franja Horaria:** La mayor cantidad de accidentes ocurren durante la Noche y la Madrugada, sumando 162 casos en cada período.
5. **Victimas por Género:** Lamentablemente se contabilizaron 28 accidentes fatales en los cuales quien conducía el vehículo era del sexo femenino.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv('accidentes_transito_limpio.csv')

plt.figure(figsize=(9, 4.5))
causas_count = df['causa_principal'].value_counts()

sns.barplot(
    x=causas_count.values,
    y=causas_count.index,
    hue=causas_count.index,
    palette='Blues_r',
    legend=False
)

plt.title('Causa de los accidentes de transito', fontsize=12, fontweight='bold')
plt.xlabel('Cantidad de Accidentes', fontsize=10)
plt.ylabel('Causa Principal', fontsize=10)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
vehiculo_fallecidos = df.groupby('tipo_vehiculo')['fallecidos'].sum().reset_index().sort_values(by='fallecidos', ascending=False)

sns.barplot(
    data=vehiculo_fallecidos,
    x='tipo_vehiculo',
    y='fallecidos',
    hue='tipo_vehiculo',
    palette='Reds_r',
    legend=False
)

plt.title('Fallecidos por tipo de vehiculo', fontsize=12, fontweight='bold')
plt.xlabel('Tipo de Vehículo', fontsize=10)
plt.ylabel('Total de Fallecidos', fontsize=10)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4.5))
dept_fallecidos = df.groupby('departamento')['fallecidos'].sum().reset_index().sort_values(by='fallecidos', ascending=False)

sns.barplot(
    data=dept_fallecidos,
    x='departamento',
    y='fallecidos',
    hue='departamento',
    palette='YlOrRd_r',
    legend=False
)

plt.title('Victimas por Departamento', fontsize=12, fontweight='bold')
plt.xlabel('Departamento', fontsize=10)
plt.ylabel('Total de Víctimas Fallecidas', fontsize=10)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
horario_order = ['Madrugada', 'Mañana', 'Tarde', 'Noche']

sns.countplot(
    data=df,
    x='rango_horario',
    order=horario_order,
    hue='rango_horario',
    palette='Purples_r',
    legend=False
)

plt.title('Accidentes por franja horaria', fontsize=12, fontweight='bold')
plt.xlabel('Franja Horaria', fontsize=10)
plt.ylabel('Cantidad de Accidentes', fontsize=10)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))
sexo_victimas = df.groupby('sexo_conductor')['total_victimas'].sum()

plt.pie(
    sexo_victimas.values,
    labels=sexo_victimas.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=['#5B9BD5', '#ED7D31']
)

plt.title('victimas por sexo del conductor', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(
    page_title="Dashboard de Accidentes de Tránsito",
    page_icon="🚗",
    layout="wide"
)

@st.cache_data
def cargar_datos():
    df = pd.read_csv('accidentes_transito_limpio.csv')
    return df

df = cargar_datos()

st.title("🚗 Dashboard de Análisis de Accidentes de Tránsito")
st.markdown("""
Esta aplicación interactiva permite consultar los indicadores clave y la distribución de
accidentes viales según filtros geográficos, causas principales, franjas horarias y sexo del conductor.
""")

st.sidebar.header("Filtros de Búsqueda")

deptos_disponibles = ['Todos'] + list(df['departamento'].unique())
depto_seleccionado = st.sidebar.selectbox("Selecciona un Departamento:", deptos_disponibles)

vehiculos_disponibles = ['Todos'] + list(df['tipo_vehiculo'].unique())
vehiculo_seleccionado = st.sidebar.selectbox("Selecciona Tipo de Vehículo:", vehiculos_disponibles)

df_filtrado = df.copy()

if depto_seleccionado != 'Todos':
    df_filtrado = df_filtrado[df_filtrado['departamento'] == depto_seleccionado]

if vehiculo_seleccionado != 'Todos':
    df_filtrado = df_filtrado[df_filtrado['tipo_vehiculo'] == vehiculo_seleccionado]

st.sidebar.markdown("---")
st.sidebar.write(f"**Registros filtrados:** {len(df_filtrado)} de {len(df)}")

col1, col2, col3, col4 = st.columns(4)

total_accidentes = len(df_filtrado)
total_lesionados = df_filtrado['lesionados'].sum()
total_fallecidos = df_filtrado['fallecidos'].sum()
total_victimas = df_filtrado['total_victimas'].sum()

col1.metric("Total de Accidentes", total_accidentes)
col2.metric("Total Lesionados", total_lesionados)
col3.metric("Total Fallecidos", total_fallecidos)
col4.metric("Total Víctimas", total_victimas)

st.markdown("---")

col_left, col_right = st.columns(2)

with col_left:
    st.subheader("1. Causa de los accidentes de transito")
    fig1, ax1 = plt.subplots(figsize=(7, 4))
    causas_count = df_filtrado['causa_principal'].value_counts()
    sns.barplot(x=causas_count.values, y=causas_count.index, hue=causas_count.index, palette="Blues_r", legend=False, ax=ax1)
    ax1.set_xlabel("Cantidad de Accidentes")
    ax1.set_ylabel("Causa Principal")
    st.pyplot(fig1)

with col_right:
    st.subheader("2. Accidentes por franja horaria")
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    horario_order = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
    sns.countplot(data=df_filtrado, x='rango_horario', order=horario_order, hue='rango_horario', palette="Purples_r", legend=False, ax=ax2)
    ax2.set_xlabel("Franja Horaria")
    ax2.set_ylabel("Cantidad de Accidentes")
    st.pyplot(fig2)

col_left2, col_right2 = st.columns(2)

with col_left2:
    st.subheader("3. Fallecidos por tipo de vehiculo")
    fig3, ax3 = plt.subplots(figsize=(7, 4))
    vehiculo_fallecidos = df_filtrado.groupby('tipo_vehiculo')['fallecidos'].sum().reset_index().sort_values(by='fallecidos', ascending=False)
    sns.barplot(data=vehiculo_fallecidos, x='tipo_vehiculo', y='fallecidos', hue='tipo_vehiculo', palette="Reds_r", legend=False, ax=ax3)
    ax3.set_xlabel("Tipo de Vehículo")
    ax3.set_ylabel("Total de Fallecidos")
    st.pyplot(fig3)

with col_right2:
    st.subheader("4. victimas por sexo del conductor")
    fig4, ax4 = plt.subplots(figsize=(6, 4))
    sexo_victimas = df_filtrado.groupby('sexo_conductor')['total_victimas'].sum()
    if len(sexo_victimas) > 0 and sexo_victimas.sum() > 0:
        ax4.pie(
            sexo_victimas.values,
            labels=sexo_victimas.index,
            autopct='%1.1f%%',
            startangle=90,
            colors=['#5B9BD5', '#ED7D31']
        )
    else:
        ax4.text(0.5, 0.5, "Sin datos para el filtro seleccionado", ha='center', va='center')
    st.pyplot(fig4)

st.markdown("---")

st.subheader("Muestra de Datos Filtrados")
st.dataframe(df_filtrado[['id_accidente', 'fecha', 'departamento', 'causa_principal', 'tipo_vehiculo', 'sexo_conductor', 'lesionados', 'fallecidos', 'total_victimas']].head(10))

In [ ]:
!pip install -q streamlit
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!streamlit run app.py & ./cloudflared tunnel --url http://localhost:8501 2>&1 | grep --line-buffered trycloudflare.com

2026-10-20T21:11:24Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-10-20T21:11:27Z INF |  https://addition-contribute-workshops-cleveland.trycloudflare.com                         |
2026-10-20 21:11:28.510 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.192.203.92:8501



# FASE 6: INTERPRETACIÓN DE RESULTADOS Y CONCLUSIONES

## 1. Respuesta a las 5 Preguntas de Investigación

1. **¿Cuál es la causa principal con mayor frecuencia de ocurrencia?**  
   La causa principal con mayor número de incidentes registrados fue no guardar distancia, acumulando un total de 124 percances dentro de la muestra.

2. **¿Qué tipo de vehículo genera un impacto más crítico en la mortalidad?**  
   Las motocicletas son el vehículo que más está presente en los accidentes y, debido a su fabricación y falta de carrocería protectora, es posible que esto repercuta en ser el medio de transporte en el cual se reportaron más personas fallecidas (55 muertes).

3. **¿Cómo se distribuye la mortalidad por departamento?**  
   El departamento de Managua registra la mayor concentración de fatalidades viales, sumando un total de 43 víctimas mortales, probablemente al ser el departamento con mayor parque vehicular y densidad de tráfico.

4. **¿En qué franja horaria ocurre la mayor concentración de accidentes?**  
   La accidentalidad se concentra fuertemente durante los rangos de Noche y Madrugada, registrando un empate idéntico de 162 accidentes en cada una de estas franjas.

5. **¿Cuántos accidentes fatales estuvieron asociados a conductoras de sexo femenino?**  
   Se contabilizaron un total de 28 accidentes fatales en los cuales la persona al volante era del sexo femenino.

---

## 2. Las 5 Conclusiones Principales

1. **El peligro en Dos Ruedas:** Las motocicletas constituyen el vehículo de mayor riesgo en la muestra, concentrando la mayor cantidad de pérdidas humanas.
2. **Riesgo Nocturno:** Los períodos de Noche y Madrugada acumulan la mayor densidad de accidentes, asociados a menor visibilidad y mayores velocidades de circulación.
3. **No se está guardando la debida Distancia de Seguridad:** El irrespeto a la distancia debida entre vehículos es el principal detonante directo de colisiones en la muestra.
4. **Concentración Geográfica:** El departamento de Managua requiere planes de intervención prioritaria al registrar el volumen más alto de fallecidos en las vías.
5. **Diferencia por Género:** La cantidad de percances mortales vinculados a conductores masculinos supera de forma considerable a la registrada en conductoras femeninas.

---

## 3. Las 3 Recomendaciones

1. **Programas de Capacitación a Motorizados:** Es de vital importancia trabajar en la concientización y en la cultura vial por parte de los conductores de motocicletas, promoviendo el uso de casco certificado y manejo defensivo.
2. **Planes profilácticos:** Incrementar la presencia policial y la fiscalización con radares en zonas céntricas y focos de mayor concentración entre zonas urbanas en las franjas de Noche y Madrugada.
3. **Controles Técnicos:** Reforzar las inspecciones mecánicas periódicas a los vehículos para prevenir fallas que desencadenen siniestros.

---

## 4. Limitaciones del Estudio

- **Dataset Simulado:** El análisis se realizó en un conjunto de datos simulado (N = 600), por lo que las métricas dependen de la distribución fijada por la semilla aleatoria y deberían validarse con registros oficiales de tránsito NAcional.


- **Horizonte Temporal Reducido:** La muestra representa un corte temporal continuo del 1 de enero al 31 de diciembre de 2025, lo cual limita la evaluación de tendencias estacionales a varios años plazo (como comparativas anuales ante feriados o temporadas de lluvia).